# (11) Jobs: betas

**Motivation**: Make $\beta$ job runnsers as txt file. <br>

In [1]:
# HIDE CODE


import os, sys
from IPython.display import display

# tmp & extras dir
git_dir = os.path.join(os.environ['HOME'], 'Dropbox/git')
extras_dir = os.path.join(git_dir, 'jb-progress-2025/_extras')
fig_base_dir = os.path.join(git_dir, 'jb-progress-2025/figs')
tmp_dir = os.path.join(git_dir, 'jb-progress-2025/tmp')

# GitHub
sys.path.insert(0, os.path.join(git_dir, '_IterativeVAE'))
# from analysis.eval import sparse_score
from figures.fighelper import *
from main.train import *

# warnings, tqdm, & style
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
from rich.jupyter import print
%matplotlib inline
set_style()

## Setup

In [2]:
from base.helper import job_runner_script


def _cleanup(path, host=None):
    for f in os.listdir(path):
        cond = f.endswith('.txt')
        if host is not None:
            cond = cond and host in f
        if cond:
            os.remove(pjoin(path, f))


def _name(host, gpu_i, fit_i):
    return f"{host}-cuda{gpu_i}-fit{fit_i}"

In [3]:
save_dir = 'Dropbox/git/_IterativeVAE/scripts'
save_dir = pjoin(os.environ['HOME'], save_dir)
os.makedirs(save_dir, exist_ok=True)

# delete existing job runners?
_cleanup(save_dir, None)

print(sorted(os.listdir(save_dir)))

['fit_model.sh', 'kill_screens.sh', 'resume_fit.sh', 'run_sessions.sh', 'test_tqdm.py', 'test_tqdm.sh']

## Betas (mach)

```<grad|lin>```

In [4]:
host = 'mach'
_cleanup(save_dir, host)

scripts_mach = collections.defaultdict(list)
tot = 0

In [5]:
# n_seeds = 5
# seeds = range(1, n_seeds + 1)

seq_len = 16
n_iters = [1, 2, 4, 8]

betas_inner = [1, 4, 8, 16]
betas = [
    8, 12, 16, 20,
    24, 32, 48, 64,
]

In [6]:
combos = itertools.product(
    enumerate(n_iters),
    betas_inner,
    betas,
)
for (gpu_i, n), b_in, b in combos:
    arg = ' '.join([
        f"--n_iters {n}",
        f"--kl_beta {b}",
        f"--beta_inner {b_in}",
        f"--comment t-{seq_len}×{n}_b-{b:0.2g},{b_in:0.2g}",
    ])
    # gpu_i = idx // 3

    kws = dict(
        device=gpu_i,
        dataset='vH16',
        archi='grad|lin',
        args=arg,
    )
    scripts_mach[gpu_i].append(job_runner_script(**kws))
    tot += 1

In [7]:
print(tot)

128

In [8]:
scripts_mach = dict(scripts_mach)
print({k: len(v) for k, v in scripts_mach.items()})

{0: 32, 1: 32, 2: 32, 3: 32}

### Save

In [9]:
n_fits = 4

for gpu_i, scripts in scripts_mach.items():
    scripts_divided = divide_list(scripts, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
        )

[PROGRESS] 'mach-cuda0-fit0.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

[PROGRESS] 'mach-cuda0-fit1.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

[PROGRESS] 'mach-cuda0-fit2.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

[PROGRESS] 'mach-cuda0-fit3.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

[PROGRESS] 'mach-cuda1-fit0.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

...

[PROGRESS] 'mach-cuda2-fit3.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

[PROGRESS] 'mach-cuda3-fit0.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

[PROGRESS] 'mach-cuda3-fit1.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

[PROGRESS] 'mach-cuda3-fit2.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

[PROGRESS] 'mach-cuda3-fit3.txt' saved at
/home/hadi/Dropbox/git/_IterativeVAE/scripts

Print one to check

In [10]:
print(combined.replace('&& ', '&& \n'))

./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 8 --beta_inner 16 --comment 
t-16×8_b-8,16 && 
./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 12 --beta_inner 16 --comment 
t-16×8_b-12,16 && 
./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 16 --beta_inner 16 --comment 
t-16×8_b-16,16 && 
./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 20 --beta_inner 16 --comment 
t-16×8_b-20,16 && 
./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 24 --beta_inner 16 --comment 
t-16×8_b-24,16 && 
./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 32 --beta_inner 16 --comment 
t-16×8_b-32,16 && 
./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 48 --beta_inner 16 --comment 
t-16×8_b-48,16 && 
./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 64 --beta_inner 16 --comment 
t-16×8_b-64,16

In [11]:
print(scripts_mach)

{
    0: [
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 8 --beta_inner 1 --comment 
t-16×1_b-8,1",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 12 --beta_inner 1 --comment 
t-16×1_b-12,1",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 16 --beta_inner 1 --comment 
t-16×1_b-16,1",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 20 --beta_inner 1 --comment 
t-16×1_b-20,1",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 24 --beta_inner 1 --comment 
t-16×1_b-24,1",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 32 --beta_inner 1 --comment 
t-16×1_b-32,1",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 48 --beta_inner 1 --comment 
t-16×1_b-48,1",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 64 --beta_inner 1 --comment 
t-16×1_b-64,1",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 8 --beta_inner 4 --comment 
t-16×1_b-8,4",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 12 --beta_inner 4 --comment 
t-16×1_b-12,4",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 16 --beta_inner 4 --comment 
t-16×1_b-16,4",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 20 --beta_inner 4 --comment 
t-16×1_b-20,4",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 24 --beta_inner 4 --comment 
t-16×1_b-24,4",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 32 --beta_inner 4 --comment 
t-16×1_b-32,4",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 48 --beta_inner 4 --comment 
t-16×1_b-48,4",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 64 --beta_inner 4 --comment 
t-16×1_b-64,4",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 8 --beta_inner 8 --comment 
t-16×1_b-8,8",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 12 --beta_inner 8 --comment 
t-16×1_b-12,8",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 16 --beta_inner 8 --comment 
t-16×1_b-16,8",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 20 --beta_inner 8 --comment 
t-16×1_b-20,8",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 24 --beta_inner 8 --comment 
t-16×1_b-24,8",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 32 --beta_inner 8 --comment 
t-16×1_b-32,8",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 48 --beta_inner 8 --comment 
t-16×1_b-48,8",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 64 --beta_inner 8 --comment 
t-16×1_b-64,8",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 8 --beta_inner 16 --comment 
t-16×1_b-8,16",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 12 --beta_inner 16 --comment
t-16×1_b-12,16",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 16 --beta_inner 16 --comment
t-16×1_b-16,16",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 20 --beta_inner 16 --comment
t-16×1_b-20,16",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 24 --beta_inner 16 --comment
t-16×1_b-24,16",
        "./fit_model.sh '0' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 1 --kl_beta 32 --beta_inner 16 --comment
t-16×1_b-32,16",
        "./fit_model.sh '0' 'v

In [12]:
print(scripts_divided)

[
    [
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 8 --beta_inner 1 --comment 
t-16×8_b-8,1",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 12 --beta_inner 1 --comment 
t-16×8_b-12,1",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 16 --beta_inner 1 --comment 
t-16×8_b-16,1",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 20 --beta_inner 1 --comment 
t-16×8_b-20,1",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 24 --beta_inner 1 --comment 
t-16×8_b-24,1",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 32 --beta_inner 1 --comment 
t-16×8_b-32,1",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 48 --beta_inner 1 --comment 
t-16×8_b-48,1",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 64 --beta_inner 1 --comment 
t-16×8_b-64,1"
    ],
    [
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 8 --beta_inner 4 --comment 
t-16×8_b-8,4",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 12 --beta_inner 4 --comment 
t-16×8_b-12,4",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 16 --beta_inner 4 --comment 
t-16×8_b-16,4",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 20 --beta_inner 4 --comment 
t-16×8_b-20,4",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 24 --beta_inner 4 --comment 
t-16×8_b-24,4",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 32 --beta_inner 4 --comment 
t-16×8_b-32,4",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 48 --beta_inner 4 --comment 
t-16×8_b-48,4",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 64 --beta_inner 4 --comment 
t-16×8_b-64,4"
    ],
    [
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 8 --beta_inner 8 --comment 
t-16×8_b-8,8",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 12 --beta_inner 8 --comment 
t-16×8_b-12,8",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 16 --beta_inner 8 --comment 
t-16×8_b-16,8",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 20 --beta_inner 8 --comment 
t-16×8_b-20,8",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 24 --beta_inner 8 --comment 
t-16×8_b-24,8",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 32 --beta_inner 8 --comment 
t-16×8_b-32,8",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 48 --beta_inner 8 --comment 
t-16×8_b-48,8",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 64 --beta_inner 8 --comment 
t-16×8_b-64,8"
    ],
    [
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 8 --beta_inner 16 --comment 
t-16×8_b-8,16",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 12 --beta_inner 16 --comment
t-16×8_b-12,16",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 16 --beta_inner 16 --comment
t-16×8_b-16,16",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 20 --beta_inner 16 --comment
t-16×8_b-20,16",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 24 --beta_inner 16 --comment
t-16×8_b-24,16",
        "./fit_model.sh '3' 'vH16' 'poisson' 'grad|lin' --seed 0 --n_iters 8 --kl_beta 32 --beta_inner 16 --comment
t-16×8_b-32,16